# Top-Tier Diabetes Prediction Model Training Pipeline (GPU Accelerated)

This notebook trains multiple machine learning models (Random Forest, XGBoost, and Logistic Regression) on the BRFSS dataset. 
It performs hyperparameter tuning to optimize strictly for **Recall (Sensitivity)** to minimize false negatives (misdiagnosis), and selects the best model for exporting.

**Hardware**: This notebook is optimized to run on an **NVIDIA GPU** using `XGBoost`'s native CUDA support and RAPIDS `cuML` (if available).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, recall_score, confusion_matrix, make_scorer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# Attempt to load GPU-accelerated RAPIDS cuML for Logistic Regression and Random Forest
# If not available, fallback to standard CPU scikit-learn
try:
    from cuml.ensemble import RandomForestClassifier
    from cuml.linear_model import LogisticRegression
    USING_CUML = True
    print("RAPIDS cuML successfully loaded. Logistic Regression and Random Forest will use NVIDIA GPU.")
except ImportError:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    USING_CUML = False
    print("RAPIDS cuML not found. Falling back to CPU scikit-learn for Logistic Regression and Random Forest. (XGBoost will still use GPU).")

## 1. Load Data
**Instructions for Teammates**: Provide the clean dataset to this step. Ensure there are no null values and junk entries (like refused surveys) are removed before splitting.

In [ ]:
# Load the dataset
# Note: Teammates should replace this with the path to the cleaned dataset.
df = pd.read_csv('diabetes_012_health_indicators_BRFSS2015.csv')

# --- [TEAMMATES DATA CLEANING PIPELINE RUNS HERE] ---
df = df.dropna()

print("Dataset shape ready for modeling:", df.shape)

## 2. Feature Matching & Target Definition
Grouping `pre-diabetes` (1) and `diabetes` (2) into a single positive class (1). 
**CRITICAL**: The features (X) must exactly match the inputs collected in the UI application.

In [ ]:
# Group pre-diabetes and diabetes together
df['Diabetes_012'] = df['Diabetes_012'].replace(2.0, 1.0)

X = df.drop('Diabetes_012', axis=1)
y = df['Diabetes_012']

# Standardize features (Important for Logistic Regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Stratified split to maintain class proportions
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Initialize SMOTE for balancing the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Class balance after SMOTE:")
print(y_train_resampled.value_counts())

## 3. Top-Tier Model Tuning (Optimizing for Recall)
We explore three powerful algorithms:
1. **Logistic Regression**: A medical standard. Provides odds ratios for feature impact.
2. **Random Forest**: Works well with SHAP for explainability and captures categorical relations.
3. **XGBoost**: State of the art algorithm, handles complex patterns and implicit class imbalance.

In [ ]:
# Create a custom scorer for Recall to use in Hyperparameter tuning
recall_scorer = make_scorer(recall_score)
models = {}

### A. Logistic Regression

In [ ]:
print("Training Logistic Regression...")

if USING_CUML:
    lr = LogisticRegression(max_iter=1000)
    lr_params = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2']}
else:
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr_params = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2'], 'solver': ['liblinear', 'saga']}

lr_search = RandomizedSearchCV(lr, lr_params, n_iter=5, scoring=recall_scorer, cv=3, random_state=42, n_jobs=-1 if not USING_CUML else 1)
lr_search.fit(X_train_resampled, y_train_resampled)

models['Logistic Regression'] = lr_search.best_estimator_

# Extract Odds Ratios for interpretability
try:
    coefs = lr_search.best_estimator_.coef_[0]
except AttributeError:
    coefs = lr_search.best_estimator_.coef_.to_numpy()[0] # Handle cuML array format if needed

odds_ratios = pd.DataFrame({
    'Feature': X.columns,
    'Odds Ratio': np.exp(coefs)
}).sort_values(by='Odds Ratio', ascending=False)

print("Top 5 Factors Increasing Risk (Odds Ratios):")
print(odds_ratios.head())

### B. Random Forest Classifier

In [ ]:
print("Training Random Forest...")

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None]
}
if not USING_CUML:
    rf_params['min_samples_split'] = [2, 5, 10]

rf = RandomForestClassifier(random_state=42) if not USING_CUML else RandomForestClassifier()
rf_search = RandomizedSearchCV(rf, rf_params, n_iter=5, scoring=recall_scorer, cv=3, random_state=42, n_jobs=-1 if not USING_CUML else 1)
rf_search.fit(X_train_resampled, y_train_resampled)

models['Random Forest'] = rf_search.best_estimator_

### C. XGBoost (GPU Accelerated)

In [ ]:
print("Training XGBoost on NVIDIA GPU...")
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# Setting tree_method='hist' and device='cuda' forces XGBoost to run on the NVIDIA GPU.
xgb_model = xgb.XGBClassifier(
    tree_method='hist', 
    device='cuda', # Use 'gpu_hist' for older XGBoost versions (< 2.0) if 'cuda' throws an error
    use_label_encoder=False, 
    eval_metric='logloss', 
    random_state=42
)

xgb_search = RandomizedSearchCV(xgb_model, xgb_params, n_iter=5, scoring=recall_scorer, cv=3, random_state=42, n_jobs=1) # Keep n_jobs=1 when using GPU to avoid CUDA context clashes
xgb_search.fit(X_train_resampled, y_train_resampled)

models['XGBoost'] = xgb_search.best_estimator_

## 4. Evaluation & Final Selection
We evaluate all optimized models on the untouched test set. The model with the best Recall (Sensitivity) will be highlighted.

In [ ]:
best_model_name = None
best_recall = 0
best_model = None

for name, model in models.items():
    print(f"\n{'='*40}")
    print(f"Evaluating {name}")
    print(f"{'='*40}")
    
    y_pred = model.predict(X_test)
    
    # Convert RAPIDS cuML cupy arrays back to numpy for sklearn metrics if necessary
    if USING_CUML and type(y_pred).__module__.startswith('cupy'):
        y_pred = y_pred.get()

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    recall = recall_score(y_test, y_pred)
    print(f"\n--> {name} Recall (Sensitivity): {recall:.4f}")
    
    if recall > best_recall:
        best_recall = recall
        best_model_name = name
        best_model = model

print(f"\n*** WINNING MODEL: {best_model_name} with Recall: {best_recall:.4f} ***")

## 5. Exporting Artifacts for the App
Save the best model AND the scaler so the application can process incoming user data exactly how the model expects it.

In [ ]:
# Export the winning model
model_filename = 'best_diabetes_model.joblib'
joblib.dump(best_model, model_filename)
print(f"Best model ({best_model_name}) exported to {model_filename}")

# Export the scaler
scaler_filename = 'scaler.joblib'
joblib.dump(scaler, scaler_filename)
print(f"StandardScaler exported to {scaler_filename} (REQUIRED for UI predictions)")